# A7-A8

In [1]:
import gmsh

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Notch dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Spacing cases [m]
# -------------------------------------------------------------------------

spacings = [50.0, 70.0]

file_index = {
    50.0: 7,
    70.0: 8,
}

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 18.0

# Local crack-tip refinement
h_refined = 2.5

refinement_x = 10.0
refinement_y = 10.0
refinement_z = 10.0

transition_thickness = 10.0

# -------------------------------------------------------------------------
# Loop over spacing cases
# -------------------------------------------------------------------------

for s in spacings:

    gmsh.initialize()

    gmsh.model.add(
        f"glacier_terminus_5C_S{s:.0f}"
    )

    # ---------------------------------------------------------------------
    # Global mesh controls
    # ---------------------------------------------------------------------

    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        h_min,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        h_max,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFactor",
        1.0,
    )

    # ---------------------------------------------------------------------
    # Glacier geometry
    # ---------------------------------------------------------------------

    glacier = gmsh.model.occ.addBox(
        0.0,
        0.0,
        0.0,
        Lx,
        Ly,
        Lz,
    )

    # ---------------------------------------------------------------------
    # Five equally spaced notches on y = 0
    #
    # Centers:
    #
    # x = Lx/2 - 2s
    # x = Lx/2 - s
    # x = Lx/2
    # x = Lx/2 + s
    # x = Lx/2 + 2s
    # ---------------------------------------------------------------------

    x_center = 0.5 * Lx

    notch_centers = [
        x_center - 2.0 * s,
        x_center - 1.0 * s,
        x_center,
        x_center + 1.0 * s,
        x_center + 2.0 * s,
    ]

    notches = []

    for xc in notch_centers:

        notch_x0 = xc - 0.5 * lx
        notch_y0 = 0.0
        notch_z0 = Lz - lz

        notch = gmsh.model.occ.addBox(
            notch_x0,
            notch_y0,
            notch_z0,
            lx,
            ly,
            lz,
        )

        notches.append(
            (3, notch)
        )

    # ---------------------------------------------------------------------
    # Subtract all five notches
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.cut(
        [(3, glacier)],
        notches,
        removeObject=True,
        removeTool=True,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Internal planes
    #
    # z = Lz / 4 = 31.25 m
    # z = Lz / 2 = 62.50 m
    # ---------------------------------------------------------------------

    z_quarter = Lz / 4.0
    z_half = Lz / 2.0

    plane_quarter = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_quarter,
        Lx,
        Ly,
    )

    plane_half = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_half,
        Lx,
        Ly,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Fragment glacier with the two horizontal planes
    #
    # This guarantees mesh nodes on:
    #
    # z = Lz/4
    # z = Lz/2
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.fragment(
        domain,
        [
            (2, plane_quarter),
            (2, plane_half),
        ],
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Physical volume
    # ---------------------------------------------------------------------

    volume_tags = [
        tag
        for dim, tag in domain
        if dim == 3
    ]

    gmsh.model.addPhysicalGroup(
        3,
        volume_tags,
        1,
    )

    gmsh.model.setPhysicalName(
        3,
        1,
        "GLACIER",
    )

    # ---------------------------------------------------------------------
    # Crack-tip refinement fields
    #
    # All five cracks start from y = 0.
    #
    # Crack tip:
    #     y = ly
    #     z = Lz - lz
    #
    # Only the region close to each crack tip is refined.
    # ---------------------------------------------------------------------

    refinement_fields = []

    crack_tip_y = ly
    crack_tip_z = Lz - lz

    for xc in notch_centers:

        field_box = gmsh.model.mesh.field.add(
            "Box"
        )

        gmsh.model.mesh.field.setNumber(
            field_box,
            "VIn",
            h_refined,
        )

        gmsh.model.mesh.field.setNumber(
            field_box,
            "VOut",
            h_max,
        )

        # -------------------------------------------------------------
        # x direction
        # -------------------------------------------------------------

        gmsh.model.mesh.field.setNumber(
            field_box,
            "XMin",
            xc - refinement_x,
        )

        gmsh.model.mesh.field.setNumber(
            field_box,
            "XMax",
            xc + refinement_x,
        )

        # -------------------------------------------------------------
        # y direction
        #
        # Refinement only around crack tip, not full Ly
        # -------------------------------------------------------------

        gmsh.model.mesh.field.setNumber(
            field_box,
            "YMin",
            max(
                0.0,
                crack_tip_y - refinement_y,
            ),
        )

        gmsh.model.mesh.field.setNumber(
            field_box,
            "YMax",
            min(
                Ly,
                crack_tip_y + refinement_y,
            ),
        )

        # -------------------------------------------------------------
        # z direction
        # -------------------------------------------------------------

        gmsh.model.mesh.field.setNumber(
            field_box,
            "ZMin",
            max(
                0.0,
                crack_tip_z - refinement_z,
            ),
        )

        gmsh.model.mesh.field.setNumber(
            field_box,
            "ZMax",
            Lz,
        )

        # -------------------------------------------------------------
        # Smooth transition
        # -------------------------------------------------------------

        gmsh.model.mesh.field.setNumber(
            field_box,
            "Thickness",
            transition_thickness,
        )

        refinement_fields.append(
            field_box
        )

    # ---------------------------------------------------------------------
    # Combine all five refinement fields
    # ---------------------------------------------------------------------

    field_min = gmsh.model.mesh.field.add(
        "Min"
    )

    gmsh.model.mesh.field.setNumbers(
        field_min,
        "FieldsList",
        refinement_fields,
    )

    gmsh.model.mesh.field.setAsBackgroundMesh(
        field_min
    )

    # ---------------------------------------------------------------------
    # Mesh settings
    # ---------------------------------------------------------------------

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        10,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFromCurvature",
        0,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFromPoints",
        0,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeExtendFromBoundary",
        0,
    )

    # ---------------------------------------------------------------------
    # Generate tetrahedral mesh
    # ---------------------------------------------------------------------

    gmsh.model.mesh.generate(3)

    # ---------------------------------------------------------------------
    # Output filename
    #
    # 07_Lx500_5C_S50.msh
    # 08_Lx500_5C_S70.msh
    # ---------------------------------------------------------------------

    index = file_index[s]

    filename = (
        f"{index:02d}/"
        f"Lx{Lx:.0f}_"
        f"5C_"
        f"S{s:.0f}.msh"
    )

    gmsh.write(filename)

    print(f"Generated: {filename}")

    gmsh.finalize()

Info    : Meshing 1D...                                                                                                              
Info    : [  0%] Meshing curve 90 (Line)
Info    : [ 10%] Meshing curve 91 (Line)
Info    : [ 10%] Meshing curve 92 (Line)
Info    : [ 10%] Meshing curve 93 (Line)
Info    : [ 10%] Meshing curve 94 (Line)
Info    : [ 10%] Meshing curve 95 (Line)
Info    : [ 10%] Meshing curve 96 (Line)
Info    : [ 10%] Meshing curve 97 (Line)
Info    : [ 10%] Meshing curve 98 (Line)
Info    : [ 20%] Meshing curve 99 (Line)
Info    : [ 20%] Meshing curve 100 (Line)
Info    : [ 20%] Meshing curve 101 (Line)
Info    : [ 20%] Meshing curve 102 (Line)
Info    : [ 20%] Meshing curve 103 (Line)
Info    : [ 20%] Meshing curve 104 (Line)
Info    : [ 20%] Meshing curve 105 (Line)
Info    : [ 20%] Meshing curve 106 (Line)
Info    : [ 20%] Meshing curve 107 (Line)
Info    : [ 30%] Meshing curve 108 (Line)
Info    : [ 30%] Meshing curve 109 (Line)
Info    : [ 30%] Meshing curve 110 (

Info    : Computing mesh sizes...
Info    : Done computing mesh sizes
Info    : Delaunay of       3023 points on   1 threads - mesh.nvert: 8803      
Info    :           -       2049 points filtered
Info    :           =        974 points added
Info    : Computing mesh sizes...
Info    : Done computing mesh sizes
Info    : Delaunay of        963 points on   1 threads - mesh.nvert: 9777      
Info    :           -        509 points filtered
Info    :           =        454 points added
Info    : Computing mesh sizes...
Info    : Done computing mesh sizes
Info    : Delaunay of        589 points on   1 threads - mesh.nvert: 10231     
Info    :           -        341 points filtered
Info    :           =        248 points added
Info    : Computing mesh sizes...
Info    : Done computing mesh sizes
Info    : Delaunay of        103 points on   1 threads - mesh.nvert: 10479     
Info    :           -         47 points filtered
Info    :           =         56 points added
Info    : Computing 

# XDMF - A7-A8

In [2]:
import meshio

mesh_files = ["07/Lx500_5C_S50.msh", "08/Lx500_5C_S70.msh"]
for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("tetra")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"tetra": cells},
        ),
    )
   

# Outline

In [3]:
import gmsh
import os

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Notch dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Spacing cases [m]
# -------------------------------------------------------------------------

spacings = [50.0, 70.0]

file_index = {
    50.0: 7,
    70.0: 8,
}

# -------------------------------------------------------------------------
# Outline mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 18.0

# -------------------------------------------------------------------------
# Loop over spacing cases
# -------------------------------------------------------------------------

for s in spacings:

    gmsh.initialize()

    gmsh.model.add(
        f"glacier_terminus_5C_S{s:.0f}_outline"
    )

    # ---------------------------------------------------------------------
    # Mesh controls
    # ---------------------------------------------------------------------

    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        h_min,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        h_max,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFactor",
        1.0,
    )

    # ---------------------------------------------------------------------
    # Glacier geometry
    # ---------------------------------------------------------------------

    glacier = gmsh.model.occ.addBox(
        0.0,
        0.0,
        0.0,
        Lx,
        Ly,
        Lz,
    )

    # ---------------------------------------------------------------------
    # Five equally spaced notches on y = 0
    # ---------------------------------------------------------------------

    x_center = 0.5 * Lx

    notch_centers = [
        x_center - 2.0 * s,
        x_center - 1.0 * s,
        x_center,
        x_center + 1.0 * s,
        x_center + 2.0 * s,
    ]

    notches = []

    for xc in notch_centers:

        notch = gmsh.model.occ.addBox(
            xc - 0.5 * lx,
            0.0,
            Lz - lz,
            lx,
            ly,
            lz,
        )

        notches.append(
            (3, notch)
        )

    # ---------------------------------------------------------------------
    # Subtract all five notches
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.cut(
        [(3, glacier)],
        notches,
        removeObject=True,
        removeTool=True,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Internal planes
    #
    # z = Lz / 4
    # z = Lz / 2
    #
    # Keep these so the outline mesh also contains the geometric edges
    # produced by the horizontal partitions.
    # ---------------------------------------------------------------------

    z_quarter = Lz / 4.0
    z_half = Lz / 2.0

    plane_quarter = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_quarter,
        Lx,
        Ly,
    )

    plane_half = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_half,
        Lx,
        Ly,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Fragment with the horizontal planes
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.fragment(
        domain,
        [
            (2, plane_quarter),
            (2, plane_half),
        ],
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Get all geometric curves
    # ---------------------------------------------------------------------

    curves = gmsh.model.getEntities(1)

    curve_tags = [
        tag
        for dim, tag in curves
    ]

    # ---------------------------------------------------------------------
    # Physical group for 1D outline
    # ---------------------------------------------------------------------

    if curve_tags:

        gmsh.model.addPhysicalGroup(
            1,
            curve_tags,
            1,
        )

        gmsh.model.setPhysicalName(
            1,
            1,
            "OUTLINE",
        )

    # ---------------------------------------------------------------------
    # Generate ONLY 1D line mesh
    # ---------------------------------------------------------------------

    gmsh.model.mesh.generate(1)

    # ---------------------------------------------------------------------
    # Output filenames
    #
    # 07/Lx500_5C_S50_outline.msh
    # 08/Lx500_5C_S70_outline.msh
    # ---------------------------------------------------------------------

    index = file_index[s]

    os.makedirs(
        f"{index:02d}",
        exist_ok=True,
    )

    filename = (
        f"{index:02d}/"
        f"Lx{Lx:.0f}_"
        f"5C_"
        f"S{s:.0f}_"
        f"outline.msh"
    )

    gmsh.write(filename)

    print(f"Generated: {filename}")

    gmsh.finalize()

Info    : Meshing 1D...                                                                                                              
Info    : [  0%] Meshing curve 90 (Line)
Info    : [ 10%] Meshing curve 91 (Line)
Info    : [ 10%] Meshing curve 92 (Line)
Info    : [ 10%] Meshing curve 93 (Line)
Info    : [ 10%] Meshing curve 94 (Line)
Info    : [ 10%] Meshing curve 95 (Line)
Info    : [ 10%] Meshing curve 96 (Line)
Info    : [ 10%] Meshing curve 97 (Line)
Info    : [ 10%] Meshing curve 98 (Line)
Info    : [ 20%] Meshing curve 99 (Line)
Info    : [ 20%] Meshing curve 100 (Line)
Info    : [ 20%] Meshing curve 101 (Line)
Info    : [ 20%] Meshing curve 102 (Line)
Info    : [ 20%] Meshing curve 103 (Line)
Info    : [ 20%] Meshing curve 104 (Line)
Info    : [ 20%] Meshing curve 105 (Line)
Info    : [ 20%] Meshing curve 106 (Line)
Info    : [ 20%] Meshing curve 107 (Line)
Info    : [ 30%] Meshing curve 108 (Line)
Info    : [ 30%] Meshing curve 109 (Line)
Info    : [ 30%] Meshing curve 110 (

Info    : [ 50%] Meshing curve 126 (Line)
Info    : [ 50%] Meshing curve 127 (Line)
Info    : [ 50%] Meshing curve 128 (Line)
Info    : [ 50%] Meshing curve 129 (Line)
Info    : [ 50%] Meshing curve 130 (Line)
Info    : [ 50%] Meshing curve 131 (Line)
Info    : [ 50%] Meshing curve 132 (Line)
Info    : [ 50%] Meshing curve 133 (Line)
Info    : [ 60%] Meshing curve 134 (Line)
Info    : [ 60%] Meshing curve 135 (Line)
Info    : [ 60%] Meshing curve 136 (Line)
Info    : [ 60%] Meshing curve 137 (Line)
Info    : [ 60%] Meshing curve 138 (Line)
Info    : [ 60%] Meshing curve 139 (Line)
Info    : [ 60%] Meshing curve 140 (Line)
Info    : [ 60%] Meshing curve 141 (Line)
Info    : [ 60%] Meshing curve 142 (Line)
Info    : [ 70%] Meshing curve 143 (Line)
Info    : [ 70%] Meshing curve 144 (Line)
Info    : [ 70%] Meshing curve 145 (Line)
Info    : [ 70%] Meshing curve 146 (Line)
Info    : [ 70%] Meshing curve 147 (Line)
Info    : [ 70%] Meshing curve 148 (Line)
Info    : [ 70%] Meshing curve 149

# Outline - XDMF

In [4]:
import meshio

mesh_files = ["07/Lx500_5C_S50_outline.msh", "08/Lx500_5C_S70_outline.msh"]

for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("line")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"line": cells},
        ),
    )
   